In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
plt.rcParams["figure.figsize"] = (24,18)

from cv_utils import *
from houghlines_process import *

In [2]:
folder_path = '/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/temp'
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
model = YOLO(os.path.join(model_path, "yolov8n_2nd_train.pt"))

data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
vid_name = 'real_test'
images_path = os.path.join(data_path, 'images' + '/' + vid_name)

for image_name in sorted(os.listdir(images_path))[::10]:
    image_path = os.path.join(images_path, image_name)#sorted(os.listdir(images_path))[8])
    rgb_image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)

    cropped_image = pitch_segment(rgb_image, visualize=False)
    orig_h, orig_w, _ = cropped_image.shape

    hsv_image = cv2.cvtColor(cv2.medianBlur(cv2.GaussianBlur(cropped_image,(5,5),0), 5), cv2.COLOR_BGR2HSV)
    hue = hsv_image[:, :, 0]
    start_index, stop_index = find_peak_range(hue, visualize=False)
    lower_green = start_index
    upper_green = stop_index

    mask_green = cv2.inRange(hue, lower_green, upper_green)

    _, rect, hull = detect_largest_contour(mask_green)

    temp = rgb_image.copy()
    cv2.polylines(temp, [hull], True, (255, 0, 0), 6)
    cv2.imwrite(os.path.join(folder_path, 'hull_' + image_name), temp)

    mask = np.zeros_like(cropped_image)
    cv2.fillPoly(mask, [hull], (255, 255, 255))

    pitch_cropped_image = cv2.bitwise_and(cropped_image, mask)

    filtered_player_image = pitch_cropped_image.copy()

    # Remove players from the image using YOLO predictions:
    results = model.predict(pitch_cropped_image)
    for result in results:
        boxes = result.boxes
        classes = result.names
        for box in boxes:
            coord = box.xyxy.cpu().detach().int().tolist()[0]
            (startX, startY, endX, endY) = coord

            # Draw black rectangle on the mask using bounding box coordinates
            cv2.rectangle(filtered_player_image, [startX, startY], [endX, endY], (0, 0, 0), -1)

    cv2.imwrite(os.path.join(folder_path, 'filtered_' + image_name), filtered_player_image)

    hls_image = cv2.cvtColor(filtered_player_image, cv2.COLOR_RGB2HLS)
    lightness = hls_image[:,:, 1]

    lightness = filtered_player_image[:, :, 2]

    lower_light = 120
    higher_light = 255

    mask_light = cv2.inRange(lightness, lower_light, higher_light)

    temp = filtered_player_image.copy()
    indices = zip(*np.where(mask_light == 255))
    for x, y in indices:
        b, g, r = temp[x, y, :].astype(np.float32)
        mean_blue = (g+r)/2
        if mean_blue - 40 <= b <= mean_blue + 25:
            mask_light[x, y] = 255
        else:
            mask_light[x, y] = 0

    # Canny Edge detection
    sigma = 0.5
    v = np.median(mask_light)
    lower = int(max(0, (1.0 - sigma) * v)) - 20
    upper = int(min(255, (1.0 + sigma) * v))

    kernel = np.ones((3, 3), np.uint8) 

    process_cropped_image = cv2.GaussianBlur(mask_light, (5, 5), 0)#cv2.medianBlur(cv2.erode(mask_light, kernel, iterations=1), 3)

    edges = cv2.Canny(process_cropped_image, lower, upper)

    kernelOpen=np.ones((1,1))
    kernelClose=np.ones((3,3))

    maskOpen=cv2.morphologyEx(edges,cv2.MORPH_OPEN,kernelOpen)
    maskClose=cv2.morphologyEx(maskOpen,cv2.MORPH_CLOSE,kernelClose)

    cv2.imwrite(os.path.join(folder_path, 'mask_light_' + image_name), mask_light)

    lines = cv2.HoughLinesP(maskClose, 1, np.pi / 180, threshold=50, minLineLength=60, maxLineGap=20)
    temp = cropped_image.copy()
    if lines is not None:
        for line in lines:
            pt1, pt2 = np.array([line[0][0], line[0][1]]), np.array([line[0][2], line[0][3]])
            _, length = angle_and_distance(pt1, pt2)
            if length > 400:
                cv2.line(temp, pt1, pt2, (0, 0, 255), 4, cv2.LINE_AA)
                final_lines.append(line)

    cv2.imwrite(os.path.join(folder_path, 'hough_' + image_name), temp)

    # Stitch the small lines into one long line
    final_lines = []
    bundler = HoughBundler(min_distance=10, min_angle=2)
    lines = bundler.process_lines(sorted(lines, key=lambda x: x[0][0]))
    temp = cropped_image.copy()
    if lines is not None:
        for line in lines:
            pt1, pt2 = np.array([line[0][0], line[0][1]]), np.array([line[0][2], line[0][3]])
            _, length = angle_and_distance(pt1, pt2)
            if length > 400:
                cv2.line(temp, pt1, pt2, (0, 0, 255), 4, cv2.LINE_AA)
                final_lines.append(line)

    cv2.imwrite(os.path.join(folder_path, 'bundled_hough_' + image_name), temp)


0: 576x1024 17 persons, 6.1ms
Speed: 4.2ms preprocess, 6.1ms inference, 2.2ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 11 persons, 3.2ms
Speed: 3.0ms preprocess, 3.2ms inference, 0.6ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 10 persons, 3.4ms
Speed: 3.1ms preprocess, 3.4ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 8 persons, 1 ball, 3.6ms
Speed: 3.3ms preprocess, 3.6ms inference, 1.5ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 22 persons, 3.5ms
Speed: 3.0ms preprocess, 3.5ms inference, 1.7ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 19 persons, 1 ball, 3.2ms
Speed: 4.1ms preprocess, 3.2ms inference, 2.3ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 15 persons, 1 ball, 3.8ms
Speed: 3.7ms preprocess, 3.8ms inference, 0.8ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 15 persons, 1 ball, 3.3ms
Speed: 3.5ms preprocess, 3.3ms inference, 